# Combining the Q3 Mediation Results Tables for AoU

In [1]:
import pandas as pd
import glob
import os

In [ ]:
!pwd

# Cox Analysis Only

### read in the data

In [ ]:
aou_cox_df

***change the pivot table back into csv if necessary***

In [5]:
# changing the pivot table back into a csv...

cols_to_fill = [
    'UKB_Description_For_Plots',
    'PRIOR',
    'Type',
    'OUTCOME',
    'MODEL'
]
ukb_cox_df[cols_to_fill] = ukb_cox_df[cols_to_fill].ffill()
# ukb_cox_df.to_csv('Q2_UKB_Cox_JULY_14_2026.csv',index=False)

### combine the tables

In [ ]:
# combine the LONG-format AoU and UKB results
combined_df = pd.concat([aou_cox_df,ukb_cox_df],axis=0)
combined_df

In [ ]:
combined_df = combined_df.drop(columns=['UKB_Description_For_Plots','Type'])
combined_df

### adding the labels

In [8]:
# add the UKB Description and Type columns (again)

# Load all the codes...

path_to_labels = ''
codes_df = pd.read_csv(f'{path_to_labels}/labels.csv') # read in the csv

In [ ]:
codes_df_subset = codes_df[['FinnGen_Phenocode','UKB_Description_For_Plots','Type']]
codes_df_subset.columns = codes_df_subset.columns.str.upper()
codes_df_subset

In [ ]:
# add codes_df_subset to the combined_df
t = pd.merge(combined_df, codes_df_subset, left_on='PRIOR', right_on='FINNGEN_PHENOCODE',how='left')
t

### saving it back into a csv

In [13]:
date='JULY_14_2026'
t.to_csv(f'FINAL_DF/Q2_Cox_AoU_and_UKB_{date}.csv',index=False)

# Q2 mediation analysis

## 1. Import mediation results summary table

In [ ]:
# import the data the full mediation_results summary table

import glob
import os
import pandas as pd

files = glob.glob("OUTPUT_FINAL_MEDIATION_FILES/*.csv")

dfs = []

for f in files:
    df = pd.read_csv(f)

    name = os.path.basename(f).replace(".csv", "")
    code, ndd = name.replace("subset_df_", "").split("_full_mediation_model_")

    df["CODE"] = code
    df["NDD"] = ndd

    dfs.append(df)

final = pd.concat(dfs, ignore_index=True)

final

In [ ]:
final.columns = final.columns.str.upper()
final['SIG']=final['P-VALUE']<0.05
final

In [ ]:
# make sure we have all the codes accounted for..
print(final['CODE'].nunique())
print(final['NDD'].nunique())

## 1a. Add the case_counts to final_df

In [ ]:
final_df = final.copy()
final_df = final_df.rename(columns={'UNNAMED: 0':'RESULT'})
final_df

In [185]:
final_df.to_csv('ALL_NDD_mediation_result.csv',index=False)

## 2. load the mediator model and the outcome_model

- extract only the 'SEX' term/result from the mediator model
- extract only the 'CODE' term/result from the outcome model

In [ ]:
# mediator model

import glob
import os
import pandas as pd

files = glob.glob("OUTPUT_INTERMEDIARY_FILES/*_mediator_model_*.csv")

dfs = []

for f in files:
    df = pd.read_csv(f)

    name = os.path.basename(f).replace(".csv", "")
    code, ndd = name.replace("subset_df_", "").split("_mediator_model_")

    df["CODE"] = code
    df["NDD"] = ndd

    dfs.append(df)

final_mediator_df = pd.concat(dfs, ignore_index=True)

final_mediator_df = final_mediator_df.rename(columns={'estimate':'ESTIMATE',
                            'p.value':'P-VALUE',
                            'term':'RESULT'})
final_mediator_df = final_mediator_df[final_mediator_df['RESULT']=='SEX']

final_mediator_df

In [ ]:
# outcome model

import glob
import os
import pandas as pd

files = glob.glob("OUTPUT_INTERMEDIARY_FILES/*_outcome_model_*.csv")

dfs = []

for f in files:
    df = pd.read_csv(f)

    name = os.path.basename(f).replace(".csv", "")
    code, ndd = name.replace("subset_df_", "").split("_outcome_model_")

    df["CODE"] = code
    df["NDD"] = ndd

    dfs.append(df)

final_outcome_df = pd.concat(dfs, ignore_index=True)

final_outcome_df = final_outcome_df.rename(columns={'estimate':'ESTIMATE',
                            'p.value':'P-VALUE',
                            'term':'RESULT'})

final_outcome_df = final_outcome_df[final_outcome_df["RESULT"] == final_outcome_df["CODE"]]

final_outcome_df

## 3. Create all the Pivot tables (full_mediation, mediator_model, and outcome_model) and then concate together.

In [ ]:
# create mediator model pivot
mediator_pivot = final_mediator_df[
    ['CODE', 'NDD', 'ESTIMATE', 'P-VALUE']
].set_index(['CODE', 'NDD'])

mediator_pivot.columns = pd.MultiIndex.from_product([
    ['Mediator Model'],
    ['ESTIMATE', 'P-VALUE']
])


mediator_pivot

In [ ]:
# create outcome model pivot
outcome_pivot = final_outcome_df[
    ['CODE', 'NDD', 'ESTIMATE', 'P-VALUE']
].set_index(['CODE', 'NDD'])

outcome_pivot.columns = pd.MultiIndex.from_product([
    ['Outcome Model'],
    ['ESTIMATE', 'P-VALUE']
])

outcome_pivot

In [ ]:
# combine full_pivot (from above), mediator pivot, and outcome pivots
final_pivot = pd.concat(
    [pivot, mediator_pivot, outcome_pivot],
    axis=1
)

final_pivot

In [ ]:
# add the case count

import glob
import os
import pandas as pd

files = glob.glob("data/PREPPED_FOR_MEDIATION/*_prepped_df_for_mediation.csv")

counts = []

for f in files:
    df = pd.read_csv(f,low_memory=False)

    name = os.path.basename(f).replace("_prepped_df_for_mediation.csv", "")

    # NDD is the first part of filename
    ndd = name.split("_")[0]

    # everything after NDD_ is the CODE
    code = name.replace(ndd + "_", "", 1)

    n_code = (df[code] == 1).sum()

    counts.append({
        "CODE": code,
        "NDD": ndd,
        "N_CODE": n_code
    })

case_counts = pd.DataFrame(counts)

case_counts

In [ ]:
case_counts

In [211]:
case_counts_pivot = case_counts.set_index(['CODE', 'NDD'])

In [ ]:
# combine full_pivot (from above), mediator pivot, and outcome pivots
final_pivot = pd.concat(
    [pivot, mediator_pivot, outcome_pivot,case_counts_pivot],
    axis=1
)

final_pivot

In [215]:
ndd='ALL_NDD'
date = 'AUG_10_2026'

In [ ]:
!pip install openpyxl
import openpyxl
# pivot.to_csv(f'{ndd}_MediationResult_{date}_PIVOT_TABLE.csv',index=False)
final_pivot.to_excel(f"{ndd}_MediationResult_{date}_PIVOT_TABLE.xlsx", index=True)